# CoursewareHubの更新

---

構築済みのCoursewareHubを、このテンプレートで構築した状態に更新します。

## 概要

以下のリリースのテンプレートで「121-CoursewareHubのセットアップ-ローカルユーザ認証」を実行して構築したCoursewareHubを、このテンプレートで構築した場合と同じ状態に更新します。

- [ocs-templates のリリース](https://github.com/nii-gakunin-cloud/ocs-templates/releases) の `25.04.0+20250904` から `25.10.0+20261023` まで

これらのリリースのテンプレートは、作業ディレクトリの `group_vars/all/defaults.yml` で auth-proxy と jupyterhub のコンテナイメージに `2025.03` を指定しています。これより前のリリースのテンプレートで構築した環境は、このNotebookの対象外です。

主な変更点を以下に示します。

|項目|更新前|更新後|
|:---|:---|:---|
|auth-proxy, jupyterhub のコンテナイメージ|`2025.03`|`2026.10.1`|
|auth-proxy のサーバ証明書のファイル名|`auth-proxy.chained.cer`, `auth-proxy.key`|`server.cer`, `server.key`|
|registry (5000番) の TLS の終端|registry コンテナ|auth-proxy コンテナの nginx|

更新の途中でCoursewareHubのサービスをすべて停止して起動しなおします。その間はCoursewareHubにアクセスできないので、利用者がいない時間帯に実行してください。起動中の利用者のサーバは停止せずに残り、更新の後も引き続き利用できます。

このNotebookは以下を対象としません。

- 「311-学認連携の設定を行う-直接学認フェデレーションを利用する」で学認連携の設定を行った環境
- IdP-proxy

### 前提条件

このNotebookは、以下の状態で実行することを前提としています。

- テンプレートが、このNotebookを含む版に更新されていること
- 構築に用いた作業ディレクトリ (「000-README」の `WORK_DIR`) が、このテンプレートのディレクトリの直下にあり、このNotebookがその作業ディレクトリで実行されていること

テンプレートを別のディレクトリに展開して作業ディレクトリを複製した場合も、作業ディレクトリをテンプレートのディレクトリの直下に置いてください。作業ディレクトリ内のテンプレートへのリンクや `ansible.cfg` は、次の章で確認して更新します。

## 作業環境の確認

作業ディレクトリが、このテンプレートを参照していることを確認します。

作業ディレクトリの `playbooks` や `scripts` などは、テンプレートのディレクトリへのシンボリックリンクです。テンプレートを別のディレクトリに展開して作業ディレクトリを複製した場合は、以前のテンプレートを参照したままになっているので、このテンプレートを参照するように張りなおします。

In [ ]:
from pathlib import Path

template_dir = Path('..').resolve()
if not (template_dir / 'notebooks').is_dir() or not (template_dir / 'playbooks').is_dir():
    raise RuntimeError(f'{template_dir} はテンプレートのディレクトリではありません。作業ディレクトリでこのNotebookを実行してください。')

# 000-README の setup_nb_workdir() が作成するリンク
for sub_dir in ['notebooks/images', 'scripts', 'template', 'playbooks', 'patches', 'plugins', 'docker']:
    src = template_dir / sub_dir
    dst = Path(Path(sub_dir).name)
    if not src.exists():
        continue
    if dst.is_symlink():
        if dst.resolve() == src:
            print(f'{dst}: OK ({src})')
            continue
        print(f'{dst}: {dst.resolve()} -> {src}')
        dst.unlink()
    elif dst.exists():
        raise RuntimeError(f'{dst.resolve()} がシンボリックリンクではありません。')
    else:
        print(f'{dst}: 作成 ({src})')
    dst.symlink_to(src, target_is_directory=True)

Ansibleの設定ファイル `ansible.cfg` のインベントリが、この作業ディレクトリの `inventory.yml` を指していることを確認します。作業ディレクトリを複製した場合は、複製元のものを指しているので書きなおします。

In [ ]:
from configparser import ConfigParser
import os

%run scripts/group.py

cfg_ansible = Path('ansible.cfg')
inventory_path = Path('inventory.yml').resolve()
if not cfg_ansible.exists() or not inventory_path.exists():
    raise RuntimeError('ansible.cfg または inventory.yml がありません。構築に用いた作業ディレクトリでこのNotebookを実行してください。')

cfg = ConfigParser(interpolation=None)
cfg.read(cfg_ansible, encoding='utf-8')
current_inventory = cfg.get('defaults', 'inventory', fallback='')
if Path(current_inventory) != inventory_path:
    print(f'inventory: {current_inventory} -> {inventory_path}')
    setup_ansible_cfg(inventory_path)
os.environ['ANSIBLE_CONFIG'] = str(cfg_ansible.resolve())

!cat ansible.cfg

> 作業ディレクトリを複製した場合、121 や 711 で指定したロゴファイル (`logo_file`) や JupyterHub の追加の設定ファイル (`jupyterhub_config_list`) のパスは、`group_vars` に複製元のパスで保存されています。このNotebookではこれらのファイルを使いませんが、後で 711 などを実行する場合は確認してください。

## 操作対象の設定

### グループ名

更新対象となるCoursewareHubを構築したときに指定したansibleのグループ名を指定します。

ansibleのグループ名を確認するために `group_vars`ファイル名の一覧を表示します。

In [ ]:
!ls -1 --hide all group_vars/

上のセルの出力結果を参考にしてグループ名を次のセルに指定してください。

In [ ]:
# (例)
# ugroup_name = 'CoursewareHub'

ugroup_name =

### チェック

指定されたグループ名が前提条件を満たしていることを確認します。

構築対象となるVCノードにアクセスできることを確認します。

In [ ]:
target_hub = f'{ugroup_name}_manager'

!ansible {ugroup_name} -m ping

UnitGroup名に対応する group_varsファイルが存在していることを確認します。

In [ ]:
!test -f group_vars/{ugroup_name}

CoursewareHubのサービスが実行されていることを確認します。

In [ ]:
cmd = "docker stack services --format '{{.Replicas}}' coursewarehub | grep '1/1' | wc -l"
!ansible {target_hub} -m shell -a "{{% raw %}} [ \$({cmd}) -eq 4 ] {{% endraw %}}"

group_varsから現在の設定値を読み込みます。

In [ ]:
import yaml

gvars_path = Path(f'group_vars/{ugroup_name}')
with gvars_path.open() as f:
    gvars = yaml.safe_load(f)

証明書の取得方式が証明書ファイルの手動配置 (`use_acme=False`) であることを確認します。

In [ ]:
if gvars.get('use_acme', False):
    raise RuntimeError('ACME による自動取得 (use_acme=True) を利用している環境は、このNotebookの対象ではありません。')

### 現在の状態の記録

更新後の確認のために、現在の状態を記録します。

サービスのコンテナイメージを表示します。

In [ ]:
cmd = "docker service ls --filter label=com.docker.stack.namespace=coursewarehub --format '{{.Name}} {{.Image}}'"
!ansible {target_hub} -a "{{% raw %}} {cmd} {{% endraw %}}"

443番 (auth-proxy) と 5000番 (registry) で返されるサーバ証明書のフィンガープリントを記録します。更新後も同じサーバ証明書が返されることを確認するために用います。

In [ ]:
import re

def strip_ansi(lines):
    # ansible の出力に含まれる色の制御文字を取り除く
    return [re.sub(r'\x1b\[[0-9;]*m', '', x) for x in lines]

def get_served_fingerprints():
    fingerprints = {}
    for port in [443, 5000]:
        out = !ansible {target_hub} -m shell -a \
            'openssl s_client -connect {{{{master_fqdn}}}}:{port} -servername {{{{master_fqdn}}}} < /dev/null 2> /dev/null \
            | openssl x509 -noout -fingerprint -sha256'
        lines = [x for x in strip_ansi(out) if 'Fingerprint=' in x]
        if len(lines) != 1:
            print('\n'.join(out))
            raise RuntimeError(f'{port}番のサーバ証明書を取得できませんでした。')
        fingerprints[port] = lines[0].split('=', 1)[1].strip()
    return fingerprints

served_fingerprints = get_served_fingerprints()
served_fingerprints

## コンテナイメージの指定の更新

作業ディレクトリの `group_vars/all/defaults.yml` は、「000-README」を実行したときにテンプレートから複製されたもので、テンプレートを更新しても変わりません。このファイルで指定しているコンテナイメージなどを、このテンプレートのものに更新します。

作業ディレクトリのファイルと、テンプレートのファイルの差分を表示します。auth-proxy と jupyterhub のコンテナイメージ (`auth_proxy_image`, `jupyterhub_image`) の行が変更されます。

In [ ]:
import difflib

defaults_path = Path('group_vars/all/defaults.yml')
template_defaults_path = template_dir / 'group_vars/all/defaults.yml'

print(''.join(difflib.unified_diff(
    defaults_path.read_text().splitlines(keepends=True),
    template_defaults_path.read_text().splitlines(keepends=True),
    fromfile=str(defaults_path), tofile=str(template_defaults_path))))

更新前のコンテナイメージが `2025.03` でない場合は、このNotebookの対象外のリリースで構築された環境なので、置き換えずに中止します。

コンテナイメージ以外の項目に差分がある場合は、作業ディレクトリのファイルが編集されている可能性があるため、置き換えずに中止します。その場合は差分を確認し、作業ディレクトリのファイルの `auth_proxy_image` と `jupyterhub_image` を上に表示されたテンプレートの値に書き換えてから、次の章に進んでください。

置き換える前のファイルは `defaults.yml.bak` として保存します。

In [ ]:
import shutil

with defaults_path.open() as f:
    work_defaults = yaml.safe_load(f)
with template_defaults_path.open() as f:
    template_defaults = yaml.safe_load(f)

image_keys = {'auth_proxy_image', 'jupyterhub_image'}
changed_keys = {
    k for k in work_defaults.keys() | template_defaults.keys()
    if work_defaults.get(k) != template_defaults.get(k)
}
if not changed_keys:
    print('更新は不要です。')
elif any(not str(work_defaults.get(k, '')).endswith(':2025.03') for k in image_keys):
    raise RuntimeError(f'更新前のコンテナイメージが 2025.03 ではありません: {[work_defaults.get(k) for k in sorted(image_keys)]}')
elif changed_keys <= image_keys:
    shutil.copy2(defaults_path, defaults_path.with_name(defaults_path.name + '.bak'))
    shutil.copyfile(template_defaults_path, defaults_path)
    print(f'更新しました: {sorted(changed_keys)}')
else:
    raise RuntimeError(f'コンテナイメージ以外の項目に差分があります: {sorted(changed_keys - image_keys)}')

更新後のコンテナイメージの指定を表示します。

In [ ]:
!ansible {target_hub} -c local -m debug -a 'msg="{{{{auth_proxy_image}}}} {{{{jupyterhub_image}}}}"'

## サーバ証明書の確認

更新後の auth-proxy は、managerノードの `certs` ディレクトリに配置されているサーバ証明書と秘密鍵を `server.cer`, `server.key` として配置して利用します。このファイルが現在利用しているサーバ証明書と同じであることを確認します。

配置されているサーバ証明書 (中間CA証明書を連結したファイルがある場合はそのファイル) のフィンガープリントが、443番で返されているサーバ証明書のものと一致することを確認します。一致しない場合は、現在利用しているサーバ証明書と秘密鍵を「741-サーバ証明書の更新」の「証明書の配置」の手順で配置しなおしてから、このセルを実行しなおしてください。

In [ ]:
out = !ansible {target_hub} -m shell -a 'chdir=certs \
    openssl x509 -noout -fingerprint -sha256 \
    -in $(ls {{{{master_fqdn}}}}.chained.cer {{{{master_fqdn}}}}.cer 2> /dev/null | head -n 1)'
lines = [x for x in strip_ansi(out) if 'Fingerprint=' in x]
if len(lines) != 1:
    print('\n'.join(out))
    raise RuntimeError('配置されているサーバ証明書を読み込めませんでした。')
placed_fingerprint = lines[0].split('=', 1)[1].strip()
print(f'配置されているサーバ証明書: {placed_fingerprint}')
print(f'443番のサーバ証明書:       {served_fingerprints[443]}')
if placed_fingerprint != served_fingerprints[443]:
    raise RuntimeError('配置されているサーバ証明書が、現在利用しているものと一致しません。')

秘密鍵がサーバ証明書に対応していることを確認します。

In [ ]:
!ansible {target_hub} -m shell -a 'chdir=certs \
    test "$(openssl x509 -noout -pubkey -in {{{{master_fqdn}}}}.cer | sha256sum)" = \
         "$(openssl pkey -pubout -in {{{{master_fqdn}}}}.key | sha256sum)"'

## 設定ファイルの更新

auth-proxy コンテナで用いるファイルと `docker-compose.yml` を更新します。

### auth-proxy の設定ファイルの配置

auth-proxy コンテナで用いるファイルを配置します。

ここで実行するplaybookが新たに配置するファイルを以下の表に示します。その他のファイルは変更されません。

|パス|用途|
|:---|:---|
|/etc/jupyterhub/nginx/certs/server.cer|nginx で利用するサーバ証明書|
|/etc/jupyterhub/nginx/certs/server.key|nginx で利用する秘密鍵|
|/etc/jupyterhub/nginx/nginx.conf.j2|auth-proxy の nginx の設定ファイルのテンプレート|

まず、実際に設定を変更する前にドライラン（チェックモード）でansibleを実行します。

In [ ]:
!ansible-playbook -l {target_hub} -CDv playbooks/setup-auth-proxy.yml || true

実際に設定ファイルの配置を行います。

In [ ]:
!ansible-playbook -l {target_hub} playbooks/setup-auth-proxy.yml

配置したファイルを確認します。

In [ ]:
!ansible {target_hub} -a 'ls -l {{{{jupyterhub_cfg_dir}}}}/nginx {{{{jupyterhub_cfg_dir}}}}/nginx/certs'

### docker-compose.yml の更新

CoursewareHubのサービスの構成を記述した `docker-compose.yml` を更新します。主に以下の項目が変更されます。

- auth-proxy, jupyterhub のコンテナイメージ
- 5000番の公開を registry から auth-proxy に移し、registry の TLS の設定を削除する
- auth-proxy の healthcheck の指定を削除する (コンテナイメージに定義されたものを利用する)
- auth-proxy に nginx の設定ファイルのテンプレートと、SimpleSAMLphp のログの出力先 (tmpfs) をマウントする

まず、実際に設定を変更する前にドライラン（チェックモード）でansibleを実行します。

In [ ]:
!ansible-playbook -l {target_hub} -CDv playbooks/deploy-docker-compose.yml || true

実際に `docker-compose.yml` を更新します。更新前のファイルは同じディレクトリにバックアップされます。

In [ ]:
!ansible-playbook -l {target_hub} playbooks/deploy-docker-compose.yml

## コンテナの更新

コンテナを起動しなおして、更新した設定とコンテナイメージを反映させます。

サービスの停止時間を短くするために、更新後のコンテナイメージを事前に取得しておきます。

In [ ]:
!ansible {target_hub} -a 'docker pull -q {{{{auth_proxy_image}}}}'
!ansible {target_hub} -a 'docker pull -q {{{{jupyterhub_image}}}}'

CoursewareHubのサービスを停止します。

5000番を公開するサービスが registry から auth-proxy に変わるため、サービスを起動したまま `docker stack deploy` で更新することはできません (`port '5000' is already in use` のエラーになります)。いったんすべてのサービスを停止します。利用者のサーバはこのスタックに含まれないため、停止せずに残ります。

In [ ]:
!ansible {target_hub} -a 'timeout 600 docker stack rm --detach=false coursewarehub'

CoursewareHubのサービスを起動します。

In [ ]:
!ansible {target_hub} -a 'chdir={{{{compose_dir}}}} \
    timeout 600 docker stack deploy --detach=false -c docker-compose.yml coursewarehub'

すべてのサービスが起動したことを確認します。CoursewareHubの4つのサービスの `REPLICAS` が `1/1` になっていない場合はエラーになります。

> jupyterhub コンテナが postgres コンテナより先に起動した場合、jupyterhub コンテナはデータベースに接続できずに一度終了し、自動的に起動しなおします。エラーになった場合は、しばらく待ってからこのセルを実行しなおしてください。

In [ ]:
!ansible {target_hub} -a 'docker service ls'
cmd = "docker stack services --format '{{.Replicas}}' coursewarehub | grep '1/1' | wc -l"
!ansible {target_hub} -m shell -a "{{% raw %}} [ \$({cmd}) -eq 4 ] {{% endraw %}}"

auth-proxyコンテナのログを表示してみます。

In [ ]:
!ansible {target_hub} -a 'docker service logs --since=30m coursewarehub_auth-proxy'

## 更新の確認

更新後のサービスが正しく動作していることを確認します。

サービスのコンテナイメージが更新されたことを確認します。

In [ ]:
cmd = "docker service ls --filter label=com.docker.stack.namespace=coursewarehub --format '{{.Name}} {{.Image}}'"
!ansible {target_hub} -a "{{% raw %}} {cmd} {{% endraw %}}"

auth-proxy コンテナの PHP のバージョンを表示します。

In [ ]:
cmd = "docker exec $(docker ps -q --filter label=com.docker.swarm.service.name=coursewarehub_auth-proxy) php -v"
!ansible {target_hub} -m shell -a '{{% raw %}} {cmd} {{% endraw %}}'

auth-proxy コンテナが、配置したサーバ証明書を利用していることを確認します。`server.cer` がない場合、auth-proxy コンテナは ACME によるサーバ証明書の取得を試みます。

In [ ]:
cmd = "docker exec $(docker ps -q --filter label=com.docker.swarm.service.name=coursewarehub_auth-proxy) readlink /etc/nginx/live-certs/server.cer"
out = !ansible {target_hub} -m shell -a '{{% raw %}} {cmd} {{% endraw %}}'
print('\n'.join(out))
if '/etc/nginx/certs/server.cer' not in [x.strip() for x in strip_ansi(out)]:
    raise RuntimeError('auth-proxy コンテナが配置したサーバ証明書を利用していません。')

443番と 5000番で、更新前と同じサーバ証明書が返されることを確認します。

In [ ]:
new_fingerprints = get_served_fingerprints()
for port in [443, 5000]:
    print(f'{port}: {new_fingerprints[port]}')
if new_fingerprints != served_fingerprints:
    raise RuntimeError(f'更新前と異なるサーバ証明書が返されています。更新前: {served_fingerprints}')

全てのノードの docker から registry にログインできることを確認します。

In [ ]:
!ansible {ugroup_name} -b -m shell -a \
    'echo {{{{registry_pass}}}} | docker login -u cwh --password-stdin {{{{master_fqdn}}}}:5000'

## CoursewareHubにアクセスする

更新したCoursewareHub環境にアクセスして正しく動作していることを確認してください。ログイン後にサーバの起動画面と Environments 画面が表示されることも確認してください。これらの画面は jupyterhub コンテナから registry に接続してコースイメージの一覧を取得します。

次のセルを実行するとCoursewareHubのアドレスを表示します。

In [ ]:
print(f'https://{gvars["master_fqdn"]}')

## 使われなくなったファイルの削除

更新前の auth-proxy が利用していたサーバ証明書と秘密鍵のファイルを削除します。CoursewareHubが正しく動作していることを確認してから実行してください。

In [ ]:
for name in ['auth-proxy.chained.cer', 'auth-proxy.key']:
    !ansible {target_hub} -b -m file -a 'path={{{{jupyterhub_cfg_dir}}}}/nginx/certs/{name} state=absent'